# 🎨 Vector Interpolation & Rotation Laboratory

An interactive laboratory for exploring advanced vector operations on neural activations:
- **Linear and Spherical Interpolation (Lerp/Slerp)**: Smoothly blend between activation vectors
- **Vector Rotation**: Rotate vectors in concept planes
- **Multi-Vector Blending**: Combine multiple vectors with various methods
- **Interpolation Paths**: Generate smooth transitions between concepts

## 🎯 What You'll Learn:
- How to smoothly transition between different concepts
- Differences between linear and spherical interpolation
- How to rotate vectors in high-dimensional activation space
- Techniques for blending multiple concepts together

## 🔧 Setup and Imports

In [1]:
# Install if needed
# !pip install torch transformers nnsight tqdm numpy matplotlib seaborn

import sys
import os
import warnings
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore', category=FutureWarning)

# Import nnsight_selfie and vector operations
from nnsight_selfie import (
    ModelAgnosticSelfie,
    InterpretationPrompt,
    print_device_info,
    get_optimal_device
)

from nnsight_selfie.vector_operations import (
    interpolate_vectors,
    rotate_vector_in_plane,
    multi_vector_interpolation,
    get_interpolation_path,
    project_and_rotate
)

# Standard imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple

torch.set_grad_enabled(False)

# Plotting setup
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("✅ Vector Interpolation & Rotation Lab initialized!")
print("🎨 Ready to explore advanced vector operations")

✅ Vector Interpolation & Rotation Lab initialized!
🎨 Ready to explore advanced vector operations


## 📥 Load Model

In [2]:
# Device detection
print("=== Device Detection ===")
print_device_info()
device = get_optimal_device()
print(f"\n🚀 Using device: {device}")

# Model configuration
MODEL_NAME = "google/gemma-2-2b-it"  # Adjust as needed

print(f"\n📥 Loading {MODEL_NAME}...")

selfie = ModelAgnosticSelfie(
    MODEL_NAME,
    dtype=torch.bfloat16,
    load_in_8bit=False
)

print(f"✅ Model loaded!")
print(f"📊 Layers: {len(selfie.layer_paths)}")
print(f"🔤 Vocab size: {len(selfie.model.tokenizer):,}")

# Create interpretation prompts
# BEST PROMPT - From Vector Arithmetic Lab (recommended for detailed interpretations)
concept_prompt = InterpretationPrompt(
    selfie.model.tokenizer,
    [
        """You are analyzing neural network activations that have been extracted from other texts and injected into positions marked with '_' in the text below. All '_' marks contain the same semantic representation - a compressed encoding derived from specific contexts in other texts.

Your task is to decode what concept, meaning, or semantic content is encoded at the '_' positions.

Here is the text:""",
        
        None,
        
        """ . That is the text with the activations injected at the '_' positions.
        
        Interpret what concept or meaning is represented at the '_' marks. Focus on:
- The most salient semantic content
- Key conceptual associations  
- Dominant thematic elements

Describe the concept clearly and directly in 1-2 sentences. Do not explain the process - just state what you interpret the activation to represent."""
    ]
)

# ALTERNATIVE SIMPLE PROMPTS (uncomment to use instead)
# concept_prompt = InterpretationPrompt(
#     selfie.model.tokenizer,
#     ["This represents the concept of ", None]
# )

# Emotion-specific prompt (optional, for emotion experiments)
emotion_prompt = InterpretationPrompt(
    selfie.model.tokenizer,
    ["This expresses the emotion of ", None]
)

print("\n✅ Setup complete!")
print("💡 Using detailed interpretation prompt from Vector Arithmetic Lab")

=== Device Detection ===
=== Device Information ===
Platform: Darwin arm64
Python: 3.13.5
PyTorch: 2.8.0
Optimal Device: mps

=== MPS Support ===
MPS Available: True
MPS Built: True

=== CUDA Support ===
CUDA Available: False


🚀 Using device: mps

📥 Loading google/gemma-2-2b-it...
Initializing model on device: mps
Model loaded successfully with 26 layers detected.
✅ Model loaded!
📊 Layers: 26
🔤 Vocab size: 256,000

✅ Setup complete!
💡 Using detailed interpretation prompt from Vector Arithmetic Lab


## 🎨 Helper Functions

In [3]:
def get_vector(text: str, token_pos: int, layer: int) -> torch.Tensor:
    """Extract activation vector from text at specified token and layer."""
    activations = selfie.get_activations(
        text,
        layer_indices=[layer],
        token_indices=[token_pos]
    )
    return activations[layer][0]


def get_concept_vector(concept: str, layer: int, use_chat_template: bool = False) -> torch.Tensor:
    """
    Extract activation vector for a concept using 'think about the {word}' pattern.
    
    Args:
        concept: The word/phrase to capture activations for
        layer: Layer index to extract from
        use_chat_template: Whether to apply chat template formatting
    
    Returns:
        Activation tensor for the concept
        
    Note:
        When use_chat_template=True, the format is:
        User: think about the {concept}
        Model: {concept} <-- captures from here
    """
    result = selfie.get_concept_activations(
        concepts=concept,
        layer_indices=[layer],
        use_chat_template=use_chat_template
    )
    return result[concept][layer]


def interpret_vector(vector: torch.Tensor, 
                    prompt: InterpretationPrompt,
                    injection_layer: int = 10,
                    max_tokens: int = 20) -> str:
    """Interpret vector using specified prompt."""
    result = selfie.interpret_vectors(
        [vector],
        prompt,
        injection_layer=injection_layer,
        max_new_tokens=max_tokens
    )[0]
    return result.strip()


def show_tokens(text: str, max_display: int = 15):
    """Display tokenization of text."""
    tokens = selfie.model.tokenizer.encode(text)
    token_strings = [selfie.model.tokenizer.decode([t]) for t in tokens]
    
    print(f"\n🔤 Tokenization ({len(tokens)} tokens):")
    for i in range(min(len(tokens), max_display)):
        print(f"  {i:2d}: '{token_strings[i]}'")
    
    if len(tokens) > max_display:
        print(f"  ... and {len(tokens) - max_display} more")
    
    return tokens, token_strings


def show_chat_template_example(concept: str):
    """Show how chat template formats the concept prompt with the word in model response."""
    user_prompt = f"think about the {concept}"
    
    # Format with assistant response containing the concept
    tokenizer = selfie.model.tokenizer
    
    if hasattr(tokenizer, 'apply_chat_template'):
        try:
            messages = [
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": concept}
            ]
            formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:
            formatted = f"User: {user_prompt}\nAssistant: {concept}"
    else:
        formatted = f"User: {user_prompt}\nAssistant: {concept}"
    
    print(f"\n📝 Concept: '{concept}'")
    print(f"\n🔄 Formatted with chat template:")
    print(formatted)
    
    # Show tokens and find where concept is
    tokens = tokenizer.encode(formatted)
    token_strings = [tokenizer.decode([t]) for t in tokens]
    concept_tokens = tokenizer.encode(concept, add_special_tokens=False)
    
    # Find concept position
    capture_pos = None
    for i in range(len(tokens) - len(concept_tokens), -1, -1):
        if tokens[i:i+len(concept_tokens)] == concept_tokens:
            capture_pos = i
            break
    
    print(f"\n🔤 Token breakdown:")
    for i, (tok_id, tok_str) in enumerate(zip(tokens, token_strings)):
        marker = " ← CAPTURE HERE" if i == capture_pos else ""
        print(f"  {i:2d}: '{tok_str}'{marker}")

print("✅ Helper functions loaded")

✅ Helper functions loaded


## 🎭 Chat Template Demo (Optional)

Before running experiments, let's see how chat templates work with concept extraction.

This section demonstrates:
- How the "think about the {word}" prompt is formatted with chat templates
- Where activations are captured (at the model's response BOS token)
- Comparison between raw and chat-templated extraction

In [ ]:
# ===== OPTIONAL: DEMO CHAT TEMPLATE =====
# Uncomment to see how chat templates work

# Show example for a single concept
# show_chat_template_example("happiness")

# Compare extraction methods
print("🎭 CHAT TEMPLATE DEMO")
print("=" * 60)

# Test concepts
test_concepts = ["king", "queen"]
test_layer = 15

print("\n📊 Extracting activations for concepts: " + ", ".join(test_concepts))

# Method 1: WITHOUT chat template (original method)
print("\n" + "-" * 60)
print("Method 1: WITHOUT Chat Template")
print("-" * 60)

activations_no_chat = selfie.get_concept_activations(
    concepts=test_concepts,
    layer_indices=[test_layer],
    use_chat_template=False  # Default behavior
)

for concept in test_concepts:
    vec = activations_no_chat[concept][test_layer]
    interp = interpret_vector(vec, concept_prompt, injection_layer=10, max_tokens=15)
    print(f"  {concept:10s}: {interp}")

# Method 2: WITH chat template
print("\n" + "-" * 60)
print("Method 2: WITH Chat Template")
print("-" * 60)

activations_with_chat = selfie.get_concept_activations(
    concepts=test_concepts,
    layer_indices=[test_layer],
    use_chat_template=True  # Enable chat templates!
)

for concept in test_concepts:
    vec = activations_with_chat[concept][test_layer]
    interp = interpret_vector(vec, concept_prompt, injection_layer=10, max_tokens=15)
    print(f"  {concept:10s}: {interp}")

print("\n✅ Chat template demo complete!")
print("\n💡 TIP: Set use_chat_template=True when using instruction-tuned models")
print("   This captures activations at the model's response position.")

In [19]:
# ===== EXPERIMENT PARAMETERS =====
# OPTION 1: Use old method with raw text (for backward compatibility)
USE_CHAT_TEMPLATE = True  # Set to True to use chat templates

if USE_CHAT_TEMPLATE:
    # NEW METHOD: Extract concepts using chat templates
    CONCEPT_A_WORD = "sad"
    CONCEPT_B_WORD = "happy"
    EXTRACTION_LAYER = 7
    INJECTION_LAYER = 7
    
    print("🔬 EXPERIMENT 1: Linear vs Spherical Interpolation")
    print("=" * 60)
    print("\n💡 Using chat-templated concept extraction")
    
    # Extract using new method
    print(f"\n🧮 Extracting concept vectors from layer {EXTRACTION_LAYER}...")
    concept_activations = selfie.get_concept_activations(
        concepts=[CONCEPT_A_WORD, CONCEPT_B_WORD],
        layer_indices=[EXTRACTION_LAYER],
        use_chat_template=True
    )
    
    vec_a = concept_activations[CONCEPT_A_WORD][EXTRACTION_LAYER]
    vec_b = concept_activations[CONCEPT_B_WORD][EXTRACTION_LAYER]
    
    print(f"  '{CONCEPT_A_WORD}' vector shape: {vec_a.shape}")
    print(f"  '{CONCEPT_B_WORD}' vector shape: {vec_b.shape}")
    
else:
    # OLD METHOD: Extract from raw text at specific token positions
    CONCEPT_A = "The king ruled the kingdom"
    CONCEPT_B = "The queen ruled the kingdom"
    
    TOKEN_POS_A = 2  # Position of "king"
    TOKEN_POS_B = 2  # Position of "queen"
    
    EXTRACTION_LAYER = 15
    INJECTION_LAYER = 10
    
    print("🔬 EXPERIMENT 1: Linear vs Spherical Interpolation")
    print("=" * 60)
    print("\n💡 Using raw text extraction (original method)")
    
    # Show tokenization
    print(f"\n📝 Concept A: '{CONCEPT_A}'")
    show_tokens(CONCEPT_A, 8)
    
    print(f"\n📝 Concept B: '{CONCEPT_B}'")
    show_tokens(CONCEPT_B, 8)
    
    # Extract vectors
    print(f"\n🧮 Extracting vectors from layer {EXTRACTION_LAYER}...")
    vec_a = get_vector(CONCEPT_A, TOKEN_POS_A, EXTRACTION_LAYER)
    vec_b = get_vector(CONCEPT_B, TOKEN_POS_B, EXTRACTION_LAYER)
    
    print(f"  Vector A shape: {vec_a.shape}")
    print(f"  Vector B shape: {vec_b.shape}")

# ===== INTERPOLATION EXPERIMENTS (same for both methods) =====
ALPHA_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]  # Interpolation steps

# Compare interpolation methods
print(f"\n🎨 Interpolating between concepts...")
print("\n" + "=" * 80)

for alpha in ALPHA_VALUES:
    print(f"\n📊 Alpha = {alpha:.2f} ({int(alpha*100)}% toward Concept B)")
    print("-" * 40)
    
    # Linear interpolation
    linear_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="linear")
    linear_interp = interpret_vector(linear_vec, concept_prompt, INJECTION_LAYER, max_tokens=50)
    
    # Spherical interpolation
    spherical_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="spherical")
    spherical_interp = interpret_vector(spherical_vec, concept_prompt, INJECTION_LAYER, max_tokens=50)
    
    print(f"  Linear:    {linear_interp}")
    print(f"  Spherical: {spherical_interp}")

print("\n✅ Experiment 1 complete!")

🔬 EXPERIMENT 1: Linear vs Spherical Interpolation

💡 Using chat-templated concept extraction

🧮 Extracting concept vectors from layer 7...
  'sad' vector shape: torch.Size([1, 2304])
  'happy' vector shape: torch.Size([1, 2304])

🎨 Interpolating between concepts...


📊 Alpha = 0.00 (0% toward Concept B)
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.89s/it]


  Linear:    **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly conveys a sense of sadness and melancholy. The use of the word "sad" and the overall tone of the text suggest
  Spherical: **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly expresses a feeling of sadness, using the word "sad" and the general context of the sentence.

📊 Alpha = 0.25 (25% toward Concept B)
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.93s/it]


  Linear:    **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly conveys a sense of sadness and melancholy. The use of the word "sad" and the overall tone of the text suggest
  Spherical: **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly conveys a sense of sadness and melancholy. The use of the word "sad" and the overall tone of the text suggest

📊 Alpha = 0.50 (50% toward Concept B)
----------------------------------------


...

  Linear:    **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly conveys a sense of sadness and melancholy. The use of the word "sad" and the overall tone of the text suggest
  Spherical: **Answer:**

The concept represented at the '_' positions is **sadness**. 

**Explanation:**

The text clearly conveys a sense of sadness and melancholy. The use of the word "sad" and the overall tone of the text suggest

📊 Alpha = 0.75 (75% toward Concept B)
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.88s/it]


  Linear:    **Answer:**

The concept represented at the '_' positions is **joyful contentment**. 


**Explanation:**

The text conveys a sense of happiness and satisfaction. The use of the word "happy" and the overall positive tone suggest a feeling
  Spherical: **Answer:**

The concept represented at the '_' positions is **joyful contentment**. 


**Explanation:**

The text conveys a sense of happiness and satisfaction. The use of the word "happy" and the overall positive tone suggest a feeling

📊 Alpha = 1.00 (100% toward Concept B)
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.88s/it]

  Linear:    **Answer:**

The concept represented at the '_' positions is **joyful contentment**. 


**Explanation:**

The text conveys a sense of happiness and satisfaction. The use of the word "happy" and the overall positive tone suggest a feeling
  Spherical: **Answer:**

The concept represented at the '_' positions is **joyful contentment**. 


**Explanation:**

The text conveys a sense of happiness and satisfaction. The use of the word "happy" and the overall positive tone suggest a feeling

✅ Experiment 1 complete!


In [ ]:
# ===== EXPERIMENT PARAMETERS =====
CONCEPT_A = "The king ruled the kingdom"
CONCEPT_B = "The queen ruled the kingdom"

TOKEN_POS_A = 2  # Position of "king"
TOKEN_POS_B = 2  # Position of "queen"

EXTRACTION_LAYER = 15
INJECTION_LAYER = 10

ALPHA_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]  # Interpolation steps

# ===== EXPERIMENT EXECUTION =====
print("🔬 EXPERIMENT 1: Linear vs Spherical Interpolation")
print("=" * 60)

# Show tokenization
print(f"\n📝 Concept A: '{CONCEPT_A}'")
show_tokens(CONCEPT_A, 8)

print(f"\n📝 Concept B: '{CONCEPT_B}'")
show_tokens(CONCEPT_B, 8)

# Extract vectors
print(f"\n🧮 Extracting vectors from layer {EXTRACTION_LAYER}...")
vec_a = get_vector(CONCEPT_A, TOKEN_POS_A, EXTRACTION_LAYER)
vec_b = get_vector(CONCEPT_B, TOKEN_POS_B, EXTRACTION_LAYER)

print(f"  Vector A shape: {vec_a.shape}")
print(f"  Vector B shape: {vec_b.shape}")

# Compare interpolation methods
print(f"\n🎨 Interpolating between concepts...")
print("\n" + "=" * 80)

for alpha in ALPHA_VALUES:
    print(f"\n📊 Alpha = {alpha:.2f} ({int(alpha*100)}% toward Concept B)")
    print("-" * 40)
    
    # Linear interpolation
    linear_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="linear")
    linear_interp = interpret_vector(linear_vec, concept_prompt, INJECTION_LAYER)
    
    # Spherical interpolation
    spherical_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="spherical")
    spherical_interp = interpret_vector(spherical_vec, concept_prompt, INJECTION_LAYER)
    
    print(f"  Linear:    {linear_interp}")
    print(f"  Spherical: {spherical_interp}")

print("\n✅ Experiment 1 complete!")

In [15]:
# ===== EXPERIMENT PARAMETERS =====
# Toggle between chat template and raw text methods
USE_CHAT_TEMPLATE = True  # Set to True to use chat templates

EXTRACTION_LAYER = 10
INJECTION_LAYER = 10

# Rotation angles (in radians)
ROTATION_ANGLES = [0, np.pi/16, np.pi/8, 3*np.pi/16, np.pi/4]
ANGLE_NAMES = ["0° (original)", "11.25°", "22.5°", "33.75°", "45°"]

# ===== EXPERIMENT EXECUTION =====
print("🔄 EXPERIMENT 2: Vector Rotation in Concept Planes")
print("=" * 60)

if USE_CHAT_TEMPLATE:
    # NEW METHOD: Use chat templates
    BASE_CONCEPT_WORD = "King"
    REFERENCE_CONCEPT_WORD = "Poverty"
    
    print(f"\n💡 Using chat-templated concept extraction")
    print(f"\n🧮 Extracting concept vectors from layer {EXTRACTION_LAYER}...")
    
    concept_activations = selfie.get_concept_activations(
        concepts=[BASE_CONCEPT_WORD, REFERENCE_CONCEPT_WORD],
        layer_indices=[EXTRACTION_LAYER],
        use_chat_template=True
    )
    
    base_vec = concept_activations[BASE_CONCEPT_WORD][EXTRACTION_LAYER]
    ref_vec = concept_activations[REFERENCE_CONCEPT_WORD][EXTRACTION_LAYER]
    
    print(f"  '{BASE_CONCEPT_WORD}' vector extracted")
    print(f"  '{REFERENCE_CONCEPT_WORD}' vector extracted")
    
else:
    # OLD METHOD: Raw text extraction
    BASE_CONCEPT = "I am feeling happy today"
    REFERENCE_CONCEPT = "I am feeling sad today"
    
    BASE_TOKEN_POS = 4  # "happy"
    REF_TOKEN_POS = 4   # "sad"
    
    print(f"\n💡 Using raw text extraction (original method)")
    
    # Show tokenization
    print(f"\n📝 Base concept: '{BASE_CONCEPT}'")
    show_tokens(BASE_CONCEPT, 10)
    
    print(f"\n📝 Reference concept: '{REFERENCE_CONCEPT}'")
    show_tokens(REFERENCE_CONCEPT, 10)
    
    # Extract vectors
    print(f"\n🧮 Extracting vectors from layer {EXTRACTION_LAYER}...")
    base_vec = get_vector(BASE_CONCEPT, BASE_TOKEN_POS, EXTRACTION_LAYER)
    ref_vec = get_vector(REFERENCE_CONCEPT, REF_TOKEN_POS, EXTRACTION_LAYER)

# Rotate and interpret
print(f"\n🔄 Rotating base vector around reference plane...")
print("\n" + "=" * 80)

for angle, name in zip(ROTATION_ANGLES, ANGLE_NAMES):
    print(f"\n📐 Rotation: {name}")
    print("-" * 40)
    
    # Rotate vector
    rotated = rotate_vector_in_plane(base_vec, ref_vec, angle)
    
    # Interpret using both prompts
    concept_interp = interpret_vector(rotated, concept_prompt, INJECTION_LAYER, max_tokens=50)
    # emotion_interp = interpret_vector(rotated, emotion_prompt, INJECTION_LAYER, max_tokens=15)
    
    print(f"  Concept: {concept_interp}")
    # print(f"  Emotion: {emotion_interp}")
    
    # Vector stats
    norm = torch.norm(rotated).item()
    print(f"  Vector norm: {norm:.1f}")

print("\n✅ Experiment 2 complete!")

🔄 EXPERIMENT 2: Vector Rotation in Concept Planes

💡 Using chat-templated concept extraction

🧮 Extracting concept vectors from layer 10...
  'King' vector extracted
  'Poverty' vector extracted

🔄 Rotating base vector around reference plane...


📐 Rotation: 0° (original)
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.94s/it]


  Concept: **Answer:**

The concept represented at the '_' positions is **leadership**. 

**Explanation:**

The text implies a discussion about leadership, likely in a context where someone is being described as a leader.
  Vector norm: 152.0

📐 Rotation: 11.25°
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.80s/it]


  Concept: **Answer:**

The concept represented at the '_' positions is **leadership**. 

**Explanation:**

The text implies a discussion about leadership, likely in a context where someone is being described as a leader.
  Vector norm: 152.0

📐 Rotation: 22.5°
----------------------------------------


...

  Concept: **Answer:**

The concept represented at the '_' positions is **authority**. 

**Explanation:**

The text implies a sense of power and control, suggesting a figure or entity with the ability to dictate or influence.
  Vector norm: 152.0

📐 Rotation: 33.75°
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.70s/it]


  Concept: **Answer:**

The concept represented at the '_' positions is **lack of resources**. 

**Explanation:**

The text implies a sense of deprivation or insufficiency, suggesting a lack of essential resources.
  Vector norm: 152.0

📐 Rotation: 45°
----------------------------------------


100%|██████████| 1/1 [00:01<00:00,  1.71s/it]

  Concept: **Answer:**

The concept represented at the '_' positions is **lack of resources**. 

**Explanation:**

The text implies a sense of deprivation or insufficiency, suggesting a lack of essential resources.
  Vector norm: 152.0

✅ Experiment 2 complete!


In [ ]:
# ===== EXPERIMENT PARAMETERS =====
BASE_CONCEPT = "I am feeling happy today"
REFERENCE_CONCEPT = "I am feeling sad today"

BASE_TOKEN_POS = 4  # "happy"
REF_TOKEN_POS = 4   # "sad"

EXTRACTION_LAYER = 4
INJECTION_LAYER = 4

# Rotation angles (in radians)
ROTATION_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
ANGLE_NAMES = ["0° (original)", "45°", "90°", "135°", "180° (opposite)"]

# ===== EXPERIMENT EXECUTION =====
print("🔄 EXPERIMENT 2: Vector Rotation in Concept Planes")
print("=" * 60)

# Show tokenization
print(f"\n📝 Base concept: '{BASE_CONCEPT}'")
show_tokens(BASE_CONCEPT, 10)

print(f"\n📝 Reference concept: '{REFERENCE_CONCEPT}'")
show_tokens(REFERENCE_CONCEPT, 10)

# Extract vectors
print(f"\n🧮 Extracting vectors from layer {EXTRACTION_LAYER}...")
base_vec = get_vector(BASE_CONCEPT, BASE_TOKEN_POS, EXTRACTION_LAYER)
ref_vec = get_vector(REFERENCE_CONCEPT, REF_TOKEN_POS, EXTRACTION_LAYER)

# Rotate and interpret
print(f"\n🔄 Rotating base vector around reference plane...")
print("\n" + "=" * 80)

for angle, name in zip(ROTATION_ANGLES, ANGLE_NAMES):
    print(f"\n📐 Rotation: {name}")
    print("-" * 40)
    
    # Rotate vector
    rotated = rotate_vector_in_plane(base_vec, ref_vec, angle)
    
    # Interpret using both prompts
    concept_interp = interpret_vector(rotated, concept_prompt, INJECTION_LAYER, max_tokens=15)
    emotion_interp = interpret_vector(rotated, emotion_prompt, INJECTION_LAYER, max_tokens=15)
    
    print(f"  Concept: {concept_interp}")
    print(f"  Emotion: {emotion_interp}")
    
    # Vector stats
    norm = torch.norm(rotated).item()
    print(f"  Vector norm: {norm:.1f}")

print("\n✅ Experiment 2 complete!")

## 🎭 Experiment 3: Multi-Vector Blending

Blend multiple concepts together using different methods.

In [ ]:
# ===== EXPERIMENT PARAMETERS =====
CONCEPTS = [
    "I feel very happy and joyful",
    "I feel somewhat sad and down",
    "I feel energetic and excited"
]

TOKEN_POSITIONS = [3, 3, 3]  # Focus tokens
CONCEPT_NAMES = ["Happy", "Sad", "Energetic"]

EXTRACTION_LAYER = 3
INJECTION_LAYER = 3

# Blending weights for weighted_sum method
BLEND_WEIGHTS = [0.5, 0.2, 0.3]  # Must sum to 1.0

# ===== EXPERIMENT EXECUTION =====
print("🎭 EXPERIMENT 3: Multi-Vector Blending")
print("=" * 60)

# Extract all vectors
vectors = []
print(f"\n📥 Extracting {len(CONCEPTS)} concept vectors...")

for i, (text, pos, name) in enumerate(zip(CONCEPTS, TOKEN_POSITIONS, CONCEPT_NAMES)):
    print(f"\n  {i+1}. {name}: '{text}'")
    show_tokens(text, 8)
    
    vec = get_vector(text, pos, EXTRACTION_LAYER)
    vectors.append(vec)
    print(f"     ✅ Extracted from token {pos}")

# Test different blending methods
print(f"\n🎨 Blending with different methods...")
print("\n" + "=" * 80)

methods = [
    ("centroid", None, "Equal weights average"),
    ("weighted_sum", BLEND_WEIGHTS, f"Custom weights {BLEND_WEIGHTS}"),
    ("sequential_lerp", [0.5, 0.5], "Sequential 50-50 blending")
]

for method, weights, description in methods:
    print(f"\n📊 Method: {method.upper()}")
    print(f"  Description: {description}")
    print("-" * 40)
    
    # Blend vectors
    blended = multi_vector_interpolation(
        vectors=vectors,
        weights=weights,
        method=method
    )
    
    # Interpret result
    concept_interp = interpret_vector(blended, concept_prompt, INJECTION_LAYER, max_tokens=20)
    emotion_interp = interpret_vector(blended, emotion_prompt, INJECTION_LAYER, max_tokens=20)
    
    print(f"  Concept: {concept_interp}")
    print(f"  Emotion: {emotion_interp}")
    
    # Stats
    flat = blended.flatten()
    print(f"  Vector norm: {torch.norm(flat):.1f}")
    print(f"  Mean: {flat.mean():.3f}, Std: {flat.std():.3f}")

print("\n✅ Experiment 3 complete!")

## 🛤️ Experiment 4: Interpolation Paths

Generate smooth paths between concepts to visualize semantic transitions.

In [ ]:
# ===== EXPERIMENT PARAMETERS =====
START_CONCEPT = "The morning was dark and gloomy"
END_CONCEPT = "The morning was bright and cheerful"

START_TOKEN_POS = 4  # "dark"
END_TOKEN_POS = 4    # "bright"

EXTRACTION_LAYER = 5
INJECTION_LAYER = 5

NUM_STEPS = 7  # Number of interpolation steps

# ===== EXPERIMENT EXECUTION =====
print("🛤️ EXPERIMENT 4: Interpolation Paths")
print("=" * 60)

# Show concepts
print(f"\n🎯 Start: '{START_CONCEPT}'")
show_tokens(START_CONCEPT, 10)

print(f"\n🎯 End: '{END_CONCEPT}'")
show_tokens(END_CONCEPT, 10)

# Extract endpoints
print(f"\n🧮 Extracting endpoints from layer {EXTRACTION_LAYER}...")
start_vec = get_vector(START_CONCEPT, START_TOKEN_POS, EXTRACTION_LAYER)
end_vec = get_vector(END_CONCEPT, END_TOKEN_POS, EXTRACTION_LAYER)

# Generate paths for both methods
print(f"\n🛤️ Generating {NUM_STEPS}-step interpolation paths...")

linear_path = get_interpolation_path(start_vec, end_vec, NUM_STEPS, method="linear")
spherical_path = get_interpolation_path(start_vec, end_vec, NUM_STEPS, method="spherical")

# Interpret each step
print("\n" + "=" * 80)
print("SEMANTIC TRANSITION PATH")
print("=" * 80)

for i, (lin_vec, sph_vec) in enumerate(zip(linear_path, spherical_path)):
    alpha = i / (NUM_STEPS - 1)
    
    print(f"\n📍 Step {i+1}/{NUM_STEPS} (α={alpha:.2f})")
    print("-" * 40)
    
    # Linear interpretation
    lin_interp = interpret_vector(lin_vec, concept_prompt, INJECTION_LAYER, max_tokens=20)
    print(f"  Linear:    {lin_interp}")
    
    # Spherical interpretation
    sph_interp = interpret_vector(sph_vec, concept_prompt, INJECTION_LAYER, max_tokens=20)
    print(f"  Spherical: {sph_interp}")

print("\n✅ Experiment 4 complete!")

## 🎯 Experiment 5: Project and Rotate

Combine projection and rotation for controlled transformations.

In [ ]:
# ===== EXPERIMENT PARAMETERS =====
MAIN_CONCEPT = "The doctor was professional and caring"
PROJECTION_AXIS = "The person showed strong emotions"

MAIN_TOKEN_POS = 2     # "doctor"
PROJ_TOKEN_POS = 4     # "strong"

EXTRACTION_LAYER = 4
INJECTION_LAYER = 4

ROTATION_ANGLE = np.pi / 2  # 90 degrees

# ===== EXPERIMENT EXECUTION =====
print("🎯 EXPERIMENT 5: Project and Rotate")
print("=" * 60)

# Show concepts
print(f"\n📝 Main concept: '{MAIN_CONCEPT}'")
show_tokens(MAIN_CONCEPT, 10)

print(f"\n📝 Projection axis: '{PROJECTION_AXIS}'")
show_tokens(PROJECTION_AXIS, 10)

# Extract vectors
print(f"\n🧮 Extracting vectors from layer {EXTRACTION_LAYER}...")
main_vec = get_vector(MAIN_CONCEPT, MAIN_TOKEN_POS, EXTRACTION_LAYER)
proj_vec = get_vector(PROJECTION_AXIS, PROJ_TOKEN_POS, EXTRACTION_LAYER)

# Project and rotate
print(f"\n🎯 Projecting and rotating by {ROTATION_ANGLE:.2f} radians ({np.degrees(ROTATION_ANGLE):.1f}°)...")

# With projection
transformed_with, info_with = project_and_rotate(
    vector=main_vec,
    projection_vec=proj_vec,
    rotation_angle=ROTATION_ANGLE,
    keep_projection=True
)

# Without projection
transformed_without, info_without = project_and_rotate(
    vector=main_vec,
    projection_vec=proj_vec,
    rotation_angle=ROTATION_ANGLE,
    keep_projection=False
)

# Display results
print("\n" + "=" * 80)
print("TRANSFORMATION RESULTS")
print("=" * 80)

print(f"\n📊 Original Vector:")
orig_interp = interpret_vector(main_vec, concept_prompt, INJECTION_LAYER)
print(f"  {orig_interp}")

print(f"\n📊 With Projection Component:")
with_interp = interpret_vector(transformed_with, concept_prompt, INJECTION_LAYER)
print(f"  {with_interp}")
print(f"  Projection magnitude: {info_with['projection_magnitude']:.2f}")

print(f"\n📊 Without Projection (Rotation Only):")
without_interp = interpret_vector(transformed_without, concept_prompt, INJECTION_LAYER)
print(f"  {without_interp}")

# Component analysis
print(f"\n🔬 Component Analysis:")
print(f"  Projection norm:  {torch.norm(info_with['projection']).item():.1f}")
print(f"  Orthogonal norm:  {torch.norm(info_with['orthogonal_component']).item():.1f}")
print(f"  Rotated norm:     {torch.norm(info_with['rotated_component']).item():.1f}")

print("\n✅ Experiment 5 complete!")

## 🎨 Visualization: Interpolation Comparison

In [ ]:
# Create visualization comparing linear vs spherical interpolation
print("🎨 Creating visualization...")

# Use vectors from Experiment 1
if 'vec_a' in locals() and 'vec_b' in locals():
    alphas = np.linspace(0, 1, 20)
    
    linear_norms = []
    spherical_norms = []
    
    for alpha in alphas:
        lin_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="linear")
        sph_vec = interpolate_vectors(vec_a, vec_b, alpha=alpha, method="spherical")
        
        linear_norms.append(torch.norm(lin_vec).item())
        spherical_norms.append(torch.norm(sph_vec).item())
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(alphas, linear_norms, 'b-', label='Linear (Lerp)', linewidth=2)
    plt.plot(alphas, spherical_norms, 'r--', label='Spherical (Slerp)', linewidth=2)
    plt.xlabel('Interpolation Factor (α)', fontsize=12)
    plt.ylabel('Vector Norm', fontsize=12)
    plt.title('Linear vs Spherical Interpolation: Vector Norms', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("✅ Visualization complete!")
    print("\nNote: Spherical interpolation maintains more consistent magnitude")
else:
    print("⚠️ Run Experiment 1 first to generate visualization")

## 🔬 Custom Experiment Sandbox

Use this space for your own vector interpolation and rotation experiments!

## 📊 Summary and Key Takeaways

### Core Vector Operations:

1. **`interpolate_vectors(vec_a, vec_b, alpha, method)`**
   - Blend two vectors smoothly
   - `method="linear"`: Simple weighted average
   - `method="spherical"`: Arc-based interpolation (better for semantic transitions)

2. **`rotate_vector_in_plane(vector, reference_vec, angle)`**
   - Rotate vector around a reference direction
   - Uses Gram-Schmidt for orthogonal basis
   - Preserves vector magnitude

3. **`multi_vector_interpolation(vectors, weights, method)`**
   - Combine 3+ vectors
   - `method="centroid"`: Equal average
   - `method="weighted_sum"`: Custom weights
   - `method="sequential_lerp"`: Pairwise blending

4. **`get_interpolation_path(vec_a, vec_b, num_steps, method)`**
   - Generate smooth transition paths
   - Useful for semantic exploration

5. **`project_and_rotate(vector, projection_vec, angle, keep_projection)`**
   - Advanced: project + rotate in one operation
   - Control which components to keep

### NEW: Chat Template Extraction

**`get_concept_activations(concepts, layer_indices, use_chat_template)`**
- Extract activations for words/phrases using "think about the {word}" pattern
- **`use_chat_template=False`** (default): Captures from last token of "think about the {word}"
- **`use_chat_template=True`**: Formats as chat and captures from the word in model response

**Chat Template Format:**
```
User: think about the happiness
Model: happiness <-- captures activation here
```

This ensures we capture the model's internal representation when it's "thinking about" the concept in its response position, which is more aligned with how instruction-tuned models were trained.

**Helper Functions:**
- `get_concept_vector(concept, layer, use_chat_template)`: Extract single concept
- `show_chat_template_example(concept)`: Debug chat template formatting and see capture position

### Key Insights:
- **Spherical interpolation** preserves vector magnitude better than linear
- **Rotation** allows exploration of orthogonal semantic directions
- **Multi-vector blending** enables complex concept combinations
- **Interpolation paths** reveal semantic transition structures
- **Chat templates** capture activations where the model generates the concept word

### When to Use Chat Templates:
✅ **Use chat templates when:**
- Working with instruction-tuned models (models ending in `-it`, `-instruct`, etc.)
- Extracting concept representations for vector arithmetic
- You want the model's "thinking about X" representation
- Working with simple word/phrase concepts

❌ **Don't use chat templates when:**
- You need fine-grained control over token positions  
- Extracting from specific contexts in longer texts
- Using base (non-instruction-tuned) models
- Backward compatibility is required

### How It Works:

**Without chat template (`use_chat_template=False`):**
```
Input: "think about the happiness"
           captures at last token ↑
```

**With chat template (`use_chat_template=True`):**
```
<bos><start_of_turn>user
think about the happiness<end_of_turn>
<start_of_turn>model
happiness <-- captures here (first token of concept in model response)
```

### Next Steps:
- Apply to your specific domain (emotions, concepts, instructions)
- Experiment with different layers (early vs late)
- Try `USE_CHAT_TEMPLATE=True` in experiments 1-2
- Compare activations with/without chat templates
- Combine with vector arithmetic from other notebooks
- Use for model steering and concept manipulation

## 📊 Summary and Key Takeaways

### Functions Overview:

1. **`interpolate_vectors(vec_a, vec_b, alpha, method)`**
   - Blend two vectors smoothly
   - `method="linear"`: Simple weighted average
   - `method="spherical"`: Arc-based interpolation (better for semantic transitions)

2. **`rotate_vector_in_plane(vector, reference_vec, angle)`**
   - Rotate vector around a reference direction
   - Uses Gram-Schmidt for orthogonal basis
   - Preserves vector magnitude

3. **`multi_vector_interpolation(vectors, weights, method)`**
   - Combine 3+ vectors
   - `method="centroid"`: Equal average
   - `method="weighted_sum"`: Custom weights
   - `method="sequential_lerp"`: Pairwise blending

4. **`get_interpolation_path(vec_a, vec_b, num_steps, method)`**
   - Generate smooth transition paths
   - Useful for semantic exploration

5. **`project_and_rotate(vector, projection_vec, angle, keep_projection)`**
   - Advanced: project + rotate in one operation
   - Control which components to keep

### Key Insights:
- **Spherical interpolation** preserves vector magnitude better than linear
- **Rotation** allows exploration of orthogonal semantic directions
- **Multi-vector blending** enables complex concept combinations
- **Interpolation paths** reveal semantic transition structures

### Next Steps:
- Apply to your specific domain (emotions, concepts, instructions)
- Experiment with different layers (early vs late)
- Combine with vector arithmetic from other notebooks
- Use for model steering and concept manipulation